# Time-Frequency Ordinal OBD-II Engine Health Model - Kaggle GPU run

Trains and evaluates the custom model (`src/custom/`) with **Leave-One-Driver-Out** cross-validation
on 3-class ordinal labels (Normal < Warning < Severe).

**Before you run:**
1. Settings -> Accelerator -> **GPU T4 x2** (or P100). One GPU is used.
2. Add Data -> the private dataset built by `kaggle/prepare_kaggle_bundle.py`
   (contains `data/obd2_engine_health_lodo.csv.gz` and `code/src/custom/`).
3. Internet on only if `xgboost` / `imbalanced-learn` need installing (both are usually preinstalled).

**What runs:** data audit -> sanity checks (tiny-overfit, label-permutation) -> LODO main model
-> ablations A0-A4 -> baselines (XGBoost, 1D-CNN, BiGRU). Everything lands in
`/kaggle/working/results` and is zipped at the end.


## 1. Environment


In [ ]:
import os, sys, subprocess, shutil, glob, json, time
import torch

print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("cuda   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device :", torch.cuda.get_device_name(0))
    print("memory : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    print("!! No GPU detected - Settings > Accelerator > GPU, then re-run.")


## 2. Run configuration


In [ ]:
# ---- what to run -------------------------------------------------------
DATASET        = "obd2"   # "obd2" = OBD-II driving data, 3-class ordinal LODO
                          # "efdb" = EngineFaultDB (kept working, not the focus)

QUICK_TEST     = False   # True -> 3 epochs per fold, to prove the plumbing works
RUN_SANITY     = True
RUN_LODO       = True    # main model, one run per CV fold
RUN_ABLATIONS  = True
RUN_BASELINES  = True    # XGBoost, 1D-CNN, BiGRU - the models you must beat
EPOCHS         = 3 if QUICK_TEST else None
SEEDS          = [42, 43, 44]   # 3 seeds; ablation gaps are small, one seed cannot rank them

# ---- EngineFaultDB settings -------------------------------------------
# split  blocked = session-aware temporal CV (report this)
#        random  = the paper's 80/20 (quote for comparability only; it leaks)
EFDB_SPLIT     = "blocked"
EFDB_FOLDS     = 5
EFDB_MERGE_23  = True    # merge the inseparable faults 2 and 3 -> 3 classes.
                         # False reproduces the published 4-class task, whose
                         # information ceiling is ~0.75.

# ---- OBD-II settings (ignored when DATASET == "efdb") ------------------
WINDOWING          = "contiguous"
VAL_SPLIT          = "interleaved"
SEGMENT_VOTE       = 0       # OFF: matches the run that produced the 0.7271
                             # ablation table. Enable (e.g. 20) only as a
                             # separate experiment - it lifts baselines too.
PER_DRIVER_SCALING = False   # OFF: also not active in that run. Changing
                             # two things at once confounds the comparison.
LABEL_SMOOTHING    = 0
DRIVER_IDS         = [1, 2, 3]
# ------------------------------------------------------------------------

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("./kaggle_run")

if DATASET == "efdb":
    tag = f"efdb_{EFDB_SPLIT}_{'3class' if EFDB_MERGE_23 else '4class'}"
    FOLD_IDS = list(range(EFDB_FOLDS))
else:
    tag = (f"obd2_{WINDOWING}_{VAL_SPLIT}_seg{SEGMENT_VOTE}"
           f"_pds{int(PER_DRIVER_SCALING)}_ls{LABEL_SMOOTHING}")
    FOLD_IDS = DRIVER_IDS

RESULTS_ROOT = os.path.join(WORK, "results", tag)

def results_dir(seed):
    return os.path.join(RESULTS_ROOT, f"seed_{seed}")

RESULTS = results_dir(SEEDS[0])
for s in SEEDS:
    os.makedirs(results_dir(s), exist_ok=True)

print(f"dataset : {DATASET}")
print(f"setup   : {tag}, seeds={SEEDS}, folds={FOLD_IDS}")
print("results ->", RESULTS_ROOT)


## 3. Locate the attached dataset

Finds the data file and the code folder anywhere under `/kaggle/input`.


In [ ]:
SEARCH_ROOTS = [os.environ.get("OBD2_BUNDLE"), "/kaggle/input", "./bundle", "../kaggle/bundle"]
SEARCH_ROOTS = [r for r in SEARCH_ROOTS if r and os.path.isdir(r)]

def find(pattern):
    hits = []
    for root in SEARCH_ROOTS:
        hits += glob.glob(os.path.join(root, pattern), recursive=True)
    return sorted(hits)

if DATASET == "efdb":
    data_hits = find("**/EngineFaultDB_Final.csv")
    need = "EngineFaultDB_Final.csv"
else:
    # Kaggle decompresses .gz while processing a dataset, so accept either form.
    data_hits = (find("**/obd2_engine_health_lodo.csv.gz")
                 or find("**/obd2_engine_health_lodo.csv")
                 or find("**/*Classified*.csv"))
    need = "obd2_engine_health_lodo.csv"

code_hits = find("**/src/custom/main.py")

if not data_hits:
    raise SystemExit(f"{need} not found. Attach the dataset built by "
                     "prepare_kaggle_bundle.py (Add Data in the right-hand panel).")
if not code_hits:
    raise SystemExit("code/src/custom/main.py not found in the attached dataset.")

DATA_FILE = data_hits[0]
SRC_CUSTOM = os.path.dirname(code_hits[0])
print("data :", DATA_FILE, "(%.1f MB)" % (os.path.getsize(DATA_FILE) / 1e6))
print("code :", SRC_CUSTOM)


## 4. Stage the code (`/kaggle/input` is read-only, so copy it into working)


In [ ]:
PROJECT = os.path.join(WORK, "obd2_model")            # stands in for the OBD2 project root
DEST = os.path.join(PROJECT, "src", "custom")
os.makedirs(os.path.dirname(DEST), exist_ok=True)
if os.path.exists(DEST):
    shutil.rmtree(DEST)
shutil.copytree(SRC_CUSTOM, DEST, ignore=shutil.ignore_patterns("__pycache__", "*.pyc"))
MAIN = os.path.join(DEST, "main.py")
CONFIG = os.path.join(DEST, "config_efdb.yaml" if DATASET == "efdb" else "config.yaml")
print("staged", len(os.listdir(DEST)), "entries in", DEST)

for mod, pkg in [("xgboost", "xgboost"), ("imblearn", "imbalanced-learn")]:
    try:
        __import__(mod)
        print(f"{mod}: present")
    except ImportError:
        print(f"{mod}: installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)


## 5. Runner helper (streams output live)


In [ ]:
def run(*flags, tag="", seed=None):
    """Invoke src/custom/main.py with the arguments for the selected dataset."""
    seed = SEEDS[0] if seed is None else seed
    cmd = [sys.executable, "-u", MAIN,
           "--config", CONFIG,
           "--data-file", DATA_FILE,
           "--output-dir", results_dir(seed),
           "--seed", str(seed),
           "--driver-ids", *[str(d) for d in FOLD_IDS]]

    if DATASET == "efdb":
        cmd += ["--dataset", "efdb", "--split", EFDB_SPLIT, "--folds", str(EFDB_FOLDS)]
        if not EFDB_MERGE_23:
            cmd.append("--no-merge")
    else:
        cmd += ["--dataset", "obd2",
                "--windowing", WINDOWING,
                "--val-split", VAL_SPLIT,
                "--label-smoothing", str(LABEL_SMOOTHING),
                "--segment-vote", str(SEGMENT_VOTE)]
        if PER_DRIVER_SCALING:
            cmd.append("--per-driver-scaling")

    if EPOCHS:
        cmd += ["--epochs", str(EPOCHS)]
    cmd += list(flags)

    print("$ " + " ".join(cmd[1:]))
    print("-" * 70)
    start = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1,
                            env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    print("-" * 70)
    print(f"{tag or 'run'}: exit={proc.returncode} in {(time.time() - start) / 60:.1f} min")
    if proc.returncode != 0:
        raise RuntimeError(f"{tag or 'run'} failed with exit code {proc.returncode}")


## 6. Data audit

Row counts and the 3-class distribution per driver. Every driver must show all three classes -
if Severe is missing the labelling step silently degraded to binary.


In [ ]:
run("--analysis-only", tag="data audit")


## 7. Sanity checks

- **Tiny-overfit**: the model must memorise 50 random samples (>90% acc) - catches broken plumbing.
- **Label-permutation**: with shuffled labels, test accuracy must fall back to ~chance (0.33) - catches leakage.


In [ ]:
if RUN_SANITY:
    run("--sanity-only", tag="sanity checks")
else:
    print("skipped")


## 8. Main experiment

LODO main model, then ablations, then baselines. On a T4 expect roughly 10-20 min for the main
model and 1-2 h for everything, depending on where early stopping kicks in.


In [ ]:
flags = ["--skip-sanity"]
if not RUN_ABLATIONS:
    flags.append("--skip-ablations")
if not RUN_BASELINES:
    flags.append("--skip-baselines")

for i, seed in enumerate(SEEDS, 1):
    print("\n" + "=" * 70)
    print(f"  SEED {seed}  ({i}/{len(SEEDS)})")
    print("=" * 70)
    if RUN_LODO:
        run(*flags, tag=f"seed {seed}: LODO + ablations + baselines", seed=seed)
    elif RUN_ABLATIONS:
        run("--ablation-only", tag=f"seed {seed}: ablations", seed=seed)
    elif RUN_BASELINES:
        run("--baselines-only", tag=f"seed {seed}: baselines", seed=seed)
    else:
        print("nothing selected")
        break


## 9. Results


In [ ]:
import pandas as pd

def fmt(mean, sd):
    return f"{mean:.4f} ± {sd:.4f}"

# ---- main model: every fold of every seed --------------------------------
rows = []
for seed in SEEDS:
    p = os.path.join(results_dir(seed), "lodo_main", "metrics", "lodo_summary.json")
    if not os.path.exists(p):
        continue
    s = json.load(open(p))
    for i, drv in enumerate(FOLD_IDS):
        rows.append({"Seed": seed, "Fold": drv,
                     "Accuracy":  s["per_fold_accuracy"][i],
                     "Macro F1":  s["per_fold_macro_f1"][i],
                     "Severe F1": (s.get("per_fold_severe_f1")
                                   or s.get("per_fold_class2_f1"))[i],
                     "QWK":       s["per_fold_qwk"][i],
                     "Ord MAE":   s["per_fold_ordinal_mae"][i]})

if rows:
    lodo = pd.DataFrame(rows)
    print("Main model - every fold, every seed:")
    display(lodo.round(4))

    print("\nBy held-out driver (mean ± sd across seeds):")
    display(lodo.groupby("Fold").agg(["mean", "std"]).round(4))

    metrics = ["Accuracy", "Macro F1", "Severe F1", "QWK", "Ord MAE"]
    print("\nOverall:", {m: fmt(lodo[m].mean(), lodo[m].std()) for m in metrics})
else:
    print("no lodo_summary.json found yet")

# ---- ablations: mean ± sd across seeds -----------------------------------
abl = []
for seed in SEEDS:
    p = os.path.join(results_dir(seed), "ablations", "ablation_comparison.csv")
    if os.path.exists(p):
        df = pd.read_csv(p)
        df["Seed"] = seed
        abl.append(df)

if abl:
    abl = pd.concat(abl, ignore_index=True)
    cols = [c for c in ["Mean Macro F1", "Mean Acc", "Mean QWK", "Mean Sev F1",
                        "Mean Ord MAE"] if c in abl.columns]
    summary = abl.groupby(["Ablation", "Name"])[cols].agg(["mean", "std"])
    print(f"\nAblations across {abl['Seed'].nunique()} seed(s) - "
          "if the sd overlaps, the ranking is not real:")
    display(summary.round(4))

# ---- baselines: mean ± sd across seeds and folds --------------------------
base = []
for seed in SEEDS:
    for d in FOLD_IDS:
        p = os.path.join(results_dir(seed), "baselines", f"fold_{d}",
                         "baseline_comparison.csv")
        if os.path.exists(p):
            df = pd.read_csv(p)
            df["Seed"], df["Fold"] = seed, d
            base.append(df)

if base:
    base = pd.concat(base, ignore_index=True)
    cols = [c for c in ["Accuracy", "Macro F1", "QWK", "Ordinal MAE", "Severe F1"]
            if c in base.columns]
    print("\nBaselines (mean ± sd across seeds and folds):")
    display(base.groupby("Model")[cols].agg(["mean", "std"]).round(4))


In [ ]:
from IPython.display import Image, display as disp

for drv in FOLD_IDS:
    p = os.path.join(RESULTS, "lodo_main", f"fold_{drv}", "visualizations", "confusion_matrix.png")
    if os.path.exists(p):
        print(f"Fold {drv}")
        disp(Image(filename=p))

sev = os.path.join(RESULTS, "lodo_main", "visualizations", "severe_error_analysis.png")
if os.path.exists(sev):
    disp(Image(filename=sev))


## 10. Fit diagnostics
Overfit / underfit status the trainer recorded per fold.


In [ ]:
for drv in FOLD_IDS:
    p = os.path.join(RESULTS, "lodo_main", f"fold_{drv}", "checkpoints", "diagnostics.json")
    if os.path.exists(p):
        d = json.load(open(p))
        print(f"driver {drv}: {d['status']}  (best epoch {d['best_epoch']}/{d['total_epochs']}, "
              f"{float(d['training_time_sec']):.0f}s)")
        for flag in d.get("flags", []):
            print("   -", flag)


## 11. Package results for download
Checkpoints are big; the zip keeps metrics, predictions and figures.


In [ ]:
import zipfile

zip_path = os.path.join(WORK, f"results_{tag}.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(RESULTS_ROOT):
        for f in files:
            if f.endswith(".pt"):          # skip checkpoints
                continue
            full = os.path.join(root, f)
            z.write(full, os.path.relpath(full, WORK))
print("wrote %s (%.1f MB)" % (zip_path, os.path.getsize(zip_path) / 1e6))
print("Grab it from the notebook's Output tab when the run finishes.")
